In [1]:
import pandas as pd

In [2]:
from Descriptive import Descriptive

In [3]:
obj=Descriptive()

In [4]:
from nsepy import get_history as gh
import datetime as dt
import yfinance as yf
stock_symbol = "RELIANCE.NS" #NSE stocks usually end with .NS
#dowload the stock data from NSE
stk_data=yf.download(stock_symbol, start="2024-05-01", end="2025-10-09")

[*********************100%***********************]  1 of 1 completed


In [5]:
stk_data.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 358 entries, 2024-05-02 to 2025-10-08
Data columns (total 5 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   (Close, RELIANCE.NS)   358 non-null    float64
 1   (High, RELIANCE.NS)    358 non-null    float64
 2   (Low, RELIANCE.NS)     358 non-null    float64
 3   (Open, RELIANCE.NS)    358 non-null    float64
 4   (Volume, RELIANCE.NS)  358 non-null    int64  
dtypes: float64(4), int64(1)
memory usage: 16.8 KB


In [6]:
print(stk_data.columns)

MultiIndex([( 'Close', 'RELIANCE.NS'),
            (  'High', 'RELIANCE.NS'),
            (   'Low', 'RELIANCE.NS'),
            (  'Open', 'RELIANCE.NS'),
            ('Volume', 'RELIANCE.NS')],
           names=['Price', 'Ticker'])


In [7]:
exog = stk_data[[('Volume', 'RELIANCE.NS')]]

In [8]:
stk_data=stk_data[["Open","High","Low","Close"]]

In [9]:
stk_data

Price,Open,High,Low,Close
Ticker,RELIANCE.NS,RELIANCE.NS,RELIANCE.NS,RELIANCE.NS
Date,,,,
2024-05-02,1454.460329,1459.721831,1446.679164,1449.075317
2024-05-03,1453.472085,1457.374970,1399.275682,1416.912964
2024-05-06,1418.395221,1422.841601,1401.103743,1402.610596
2024-05-07,1399.102859,1403.820987,1375.413559,1384.775635
2024-05-08,1380.848112,1415.875659,1380.848112,1401.647339
...,...,...,...,...
2025-10-01,1360.708629,1372.255218,1356.428371,1362.400757
2025-10-03,1356.926092,1365.287457,1350.655159,1357.125244


In [10]:
from sklearn.preprocessing import MinMaxScaler
scaler1 = MinMaxScaler()
data1 = pd.DataFrame(scaler1.fit_transform(stk_data), columns=['Open','High','Low','Close'])
scaler2 = MinMaxScaler()
exog = pd.DataFrame(scaler2.fit_transform(exog), columns=['Volume'])

In [11]:
#DataFrame containing stock prices and a list of column names to use
data1=pd.DataFrame(data1,columns=["Open","High","Low","Close"])

In [12]:
#Training and test set split
training_size = round(len(data1 ) * 0.80)
print(training_size)
X_train=data1[:training_size]
X_test=data1[training_size:]
print("X_train length:",X_train.shape)
print("X_test length:",X_test.shape)
y_train=data1[:training_size]
y_test=data1[training_size:]
print("y_train length:",y_train.shape)
print("y_test length:",y_test.shape)

286
X_train length: (286, 4)
X_test length: (72, 4)
y_train length: (286, 4)
y_test length: (72, 4)


In [13]:
from sklearn.metrics import mean_squared_error
#Imports the function to calculate Mean Squared Error (MSE)
from sklearn.metrics import mean_absolute_percentage_error
#Imports the function to calculate MAPE (Mean Absolute Percentage Error)
import numpy as np

In [14]:
from statsmodels.tsa.statespace.varmax import VARMAX
#Imports the VARMAX (Vector Autoregressive Moving Average with exogenous variables) to create a model multiple variables together

In [ ]:
def cominbation(dataset, exog, listt):
#Defines a function named combination
    performance = { "Model": [], "RMSE": [], "MaPe": []}
    print(listt)
    #Prints the selected columns
    datasetTwo = dataset[listt]
    #Selects only the columns listed in listt
    test_obs = 28
    #Specifies that the last 28 observations will be used as the test set
    train = datasetTwo[:-test_obs]
    #Creates the training dataset,it uses all rows except the last 28
    test = datasetTwo[-test_obs:]
    #Creates the test dataset,it uses all rows except the last 28
    #Split exogenous into train_exog/test_exog
    train_exog = exog[:-test_obs]
    test_exog = exog[-test_obs:]
    # Try different (p,q) combinations
    best_aic = float("inf")
    best_order = None
    best_model = None

    for p in range(1, 4):
        #p = autoregressive (AR) order
        for q in range(0, 3):
            #q = moving average (MA) order
            try:
                model = VARMAX(train, exog=train_exog, order=(p,q))
                #Creates a VARMAX model using the training data
                result = model.fit(disp=False)

                print(f"Order ({p},{q})")
                print("AIC:", result.aic)
                #AIC (Akaike Information Criterion): lower is better
                print("BIC:", result.bic)
                #BIC (Bayesian Information Criterion): lower is better
               
                if result.aic < best_aic:
                #find the best VARMA model based on the lowest AIC value
                    best_aic = result.aic
                    #If the current model has a lower AIC, save its AIC value
                    best_order = (p, q)
                    #Save the (p, q) values of the current best VARMA model
                    best_model = result
                    #Store the complete fitted model object.
            except Exception as e:
                #any error that occurs,Instead of stopping the entire program, Python jumps to the except block 
                print(f"({p},{q}) failed:", e)
                #The variable e stores the error message
    print("\nBest Order:", best_order)

    # Forecast
    pred = best_model.forecast(steps=test_obs, exog=test_exog)
    # best_model is the VARMA model with the lowest AIC & steps=test_obs tells the model how many future time steps to predict
    preds = pd.DataFrame(pred, columns=listt)
    # Converts the forecast into a Pandas DataFrame
    preds.to_csv(f"varma_forecasted_{test_obs}.csv")
    # Saves the forecasted values to a CSV file & This file can later be opened in Excel
    from sklearn.metrics import mean_squared_error
    # Imports the function used to calculate the Mean Squared Error (MSE), which is then used to compute RMSE
    from sklearn.metrics import mean_absolute_percentage_error
    # Imports the function used to calculate the Mean Absolute Percentage Error (MAPE)
    rmse = np.sqrt(mean_squared_error(test, pred))
    # RMSE measures the average prediction error in the same units as the data and Smaller RMSE indicates better predictions
    mape = mean_absolute_percentage_error(test, pred)
    #It gives the average percentage error and Lower MAPE means better forecasting accuracy
    performance["Model"].append(listt)
    performance["RMSE"].append(rmse)
    performance["MaPe"].append(mape)
    performance["Lag"].append(best_order)
    performance["Test"].append(test_obs)
    #Stores the variables used in the model
    perf = pd.DataFrame(performance)
    #Converts the performance dictionary into a Pandas DataFrame
    return perf, best_model, pred
    #Return the results of perf, best_model, pred

In [ ]:
listt=["Close","High","Open","Low"]

In [ ]:
perf, result, pred = cominbation(data1, exog, listt)

['Close', 'High', 'Open', 'Low']


C:\Users\manju\anaconda3\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


Order (1,0)
AIC: -6650.860058575081
BIC: -6521.690908323423


C:\Users\manju\anaconda3\lib\site-packages\statsmodels\tsa\statespace\varmax.py:160: EstimationWarning: Estimation of VARMA(p,q) models is not generically robust, due especially to identification issues.
  warn('Estimation of VARMA(p,q) models is not generically robust,'


Order (1,1)
AIC: -6621.778581733936
BIC: -6431.82394901091


...

C:\Users\manju\anaconda3\lib\site-packages\statsmodels\tsa\statespace\varmax.py:160: EstimationWarning: Estimation of VARMA(p,q) models is not generically robust, due especially to identification issues.
  warn('Estimation of VARMA(p,q) models is not generically robust,'
C:\Users\manju\anaconda3\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


Order (2,2)
AIC: -6550.321543167907
BIC: -6238.795945502145


C:\Users\manju\anaconda3\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


Order (3,0)
AIC: -6595.171523403712
BIC: -6344.431408209317


C:\Users\manju\anaconda3\lib\site-packages\statsmodels\tsa\statespace\varmax.py:160: EstimationWarning: Estimation of VARMA(p,q) models is not generically robust, due especially to identification issues.
  warn('Estimation of VARMA(p,q) models is not generically robust,'


In [ ]:
data1

In [ ]:
perf